In [17]:
import sys
sys.path.append(r"C:\Users\Rakesh\Documents\summer-project")
sys.path.append(r"C:\Users\Rakesh\Documents\summer-project\src")

from functions_v2 import *
from two_level_mc import TwoLevelSetup
from graph_mlmc_model import GraphTwoLevelModel
from mlmc_runner import MLMCRunner
from sksparse.cholmod import cholesky as sparse_cholesky

In [8]:
edges, n_vertices, edge_weights = load_graph(r"../../data/raw/power-US-Grid.mtx")
num_components = check_connectivity(edges, n_vertices)
if num_components > 1:
    edges, n_vertices, edge_weights = filter_to_largest_component(edges, n_vertices, edge_weights)

A, D, L = build_graph_matrices(edges, n_vertices)
lambda_min = compute_lambda_min(L, D)
L_sigma = build_shifted_laplacian(L, D, lambda_min)

import networkx as nx
G_nx = nx.Graph()
G_nx.add_edges_from(edges)
gamma_in, gamma_out, diameter = find_diameter_endpoints(G_nx, n_sample=5, k_hop=4)

✓ Loaded: ../../data/raw/power-US-Grid.mtx
  Vertices : 4941
  Edges    : 6594
  Weighted : no

Components: 1
  -> Graph is fully connected, safe to proceed

✓ Phase 1 complete: A, D, L built as sparse matrices
  Matrix size : 4941 x 4941
  Degree range: [1, 19]
  Non-zeros in L: 18129

✓ lambda_min = 0.000271
  (eigenvalues found: [0.         0.00027102])
✓ Phase 2 complete: L_sigma built (sigma^2 = 0.25)
  L_sigma type: sparse



In [20]:
setup = TwoLevelSetup.build(
    edges, n_vertices, L_sigma, lambda_min,
    gamma_in, gamma_out,
    build_incidence_matrix, sparse_cholesky,
    max_size=10
)

gamma_in: 23, gamma_out: 14, n_vertices: 4941
Boundary fraction: 0.7488%
  -> Likely safe for aggregation (comparable to validated successes).
n_coarse: 2074 (2074 aggregates)
Size distribution -- min: 2, max: 10, mean: 2.38
Singletons: 0 (0.0%)
gamma_in_coarse: 15, gamma_out_coarse: 7
Overlap (must be empty): set()
Interior coarse vertices: 2052 (98.9%)
coarse edges: 3282 (from 6594 fine edges)


In [23]:
model = GraphTwoLevelModel(setup)
runner = MLMCRunner(model, base_seed=0)
result = runner.run_fixed(samples_per_level=[2000, 300])

print(f"Estimate: {result.estimate:.6f}")
print(f"Standard error: {result.standard_error:.6f}")

for lr in result.level_results:
    print(f"level={lr.level}, n={lr.sample_count}, mean_correction={lr.mean_correction:.6f}, "
          f"mean_cost={lr.mean_sample_cost:.6f}")

Estimate: 0.152189
Standard error: 0.000071
level=0, n=2000, mean_correction=0.236068, mean_cost=0.021848
level=1, n=300, mean_correction=-0.083880, mean_cost=0.051779
